## Model 2 Add Memory/Decay Parameter

Model 1 assumes that once the agent learns a value, it is stored perfectly until the next update. But in reality, people may forget what they learned over time. Model 2 tests this idea by introducing a parameter A.

$$V_i^{(t+1)} = A \cdot V_i^{(t)} + \alpha \cdot (o^{(t)} - V_i^{(t)})$$

- $A$ = 1 means identical to Model 1, values are perfectly preserved (no forgetting)
- $A$ < 1 means values decay toward zero between trials, the agent gradually forgets (recent experiences dominate)
- $A$ > 1 means values grow over time (generally unrealistic)
  
### Why is this relevant to our research question?

If anxious participants show lower A, it would suggest they lose learned safety information faster. This is consistent with clinical observations that anxious individuals struggle to maintain the benefits of exposure therapy over time.

In [ ]:
def simulate_model2(alpha, beta, A, n_trials=N_TRIALS, v0=V0): # Simulate Model 2 with decay parameter A
    Va, Vb = v0, v0
    ch, ou = [], []
    for t in range(n_trials):
        p_a = np.exp(-beta*Va) / (np.exp(-beta*Va) + np.exp(-beta*Vb))
        c = 1 if np.random.rand() < p_a else 0
        block = min(t // BLOCK_SIZE, 3)
        o = 1 if np.random.rand() < (PROB_A[block] if c==1 else 1-PROB_A[block]) else 0
        if c==1: Va = A*Va + alpha*(o - Va)
        else:    Vb = A*Vb + alpha*(o - Vb)
        ch.append(c); ou.append(o)
    return np.array(ch), np.array(ou)


def nll_model2(params, choices, outcomes):
    # NLL for Model 2
    alpha, beta, A = params
    Va, Vb = V0, V0
    nll = 0.0
    for t in range(len(choices)):
        exp_a = np.exp(-beta*Va); exp_b = np.exp(-beta*Vb)
        p_a = exp_a / (exp_a + exp_b)
        nll -= np.log(max(p_a, 1e-12)) if choices[t]==1 else np.log(max(1-p_a, 1e-12))
        if choices[t]==1: Va = A*Va + alpha*(outcomes[t] - Va)
        else:             Vb = A*Vb + alpha*(outcomes[t] - Vb)
    return nll

print("Model 2() defined")

In [ ]:
# Fit Model 2
x0_m2 = [0.4, 5, 0.5]
fitted_m2 = np.array([fit_model(nll_model2, choices[i], outcomes[i], x0_m2).x
                       for i in range(N_PARTICIPANTS)])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = ["tab:red"]*25 + ["tab:blue"]*25
for idx, name in enumerate(["α", "β", "A"]):
    axes[idx].bar(range(N_PARTICIPANTS), fitted_m2[:, idx], color=colors, alpha=0.7)
    axes[idx].axvline(24.5, color="black", ls="--", lw=1); axes[idx].set_xlabel("Participant")
    axes[idx].set_ylabel(name); axes[idx].set_title(f"Model 2: {name}")
plt.tight_layout(); plt.savefig("figures/task_h.png"); plt.show()

In [ ]:
# Group comparison
print("Model 2 Group comparison")
for idx, name in enumerate(["α", "β", "A"]):
    t, p = ttest_ind(fitted_m2[:25, idx], fitted_m2[25:, idx], equal_var=False)
    sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "n.s."
    print(f"  {name}:  t = {t:.4f},  p = {p:.4f}  {sig}")